## AutoShop Multi-Agent mit OpenAI Agents SDK und Agent Skills

Dieses Notebook basiert auf dem [AutoShop MCP-Server](../05-mcp/30-autoshop-mcp-server.ipynb), der zuerst gestartet werden muss.

Die bestehenden Agenten bleiben erhalten, ergänzt um **Agent Skills** nach [agentskills.io](https://agentskills.io/specification):

1. `MCPServerStreamableHttp` bindet den bestehenden AutoShop-MCP-Server ein.
2. Ein interner `Agent` verwendet freigegebene, lesende MCP-Tools für echte Fahrzeugdaten.
3. Ein externer `Agent` ergänzt nur unverbindliche allgemeine Informationen.
4. Ein Offert-Agent erstellt aus den Resultaten die Markdown-Offerte.
5. **Agent Skills** kapseln die fachlichen Abläufe in wiederverwendbaren `SKILL.md`-Dateien. Der Agent sieht zunächst nur Name und Beschreibung; die vollständige Anleitung wird erst bei Bedarf geladen.
6. `Runner.run()` übernimmt weiterhin den Agenten-/Tool-Loop.

**Wichtig:** Dieses Notebook verwendet eine kleine, lokale Skill-Integration über `@function_tool`, damit der bisherige `Agent` und der bestehende MCP-Server unverändert zusammenarbeiten. Die native SDK-Capability `Skills` gehört zur alternativen `SandboxAgent`-Architektur.

`order_delete` bleibt über die MCP-Tool-Allowlist gesperrt.

- - -

### Umgebung und Verbindung

Die bestehende `~/data/env-mcp.py` wird mit `%run` geladen. Erwartet werden:

- `OPENAI_API_KEY`
- `AI_MODEL`
- `AI_BASE_URL` (für deinen OpenAI-kompatiblen Endpoint)
- `MCP_URL` (URL des AutoShop-MCP-Servers)

Die MCP-URL wird direkt aus `env.py` übernommen; eine Kubernetes-Abfrage ist nicht erforderlich.

In [ ]:
%run ~/data/env-mcp.py

from openai import AsyncOpenAI

from agents import (
    Agent,
    Runner,
    set_default_openai_client,
    set_tracing_disabled,
)
from agents.mcp import (
    MCPServerStreamableHttp,
    create_static_tool_filter,
)

# OPENAI_API_KEY, AI_MODEL, AI_BASE_URL und MCP_URL kommen aus env.py.
# Das Agents SDK verwendet einen asynchronen OpenAI-Client.
openai_client = AsyncOpenAI(
    api_key=OPENAI_API_KEY,
    base_url=AI_BASE_URL,
)
set_default_openai_client(openai_client, use_for_tracing=False)

# Für OpenAI-kompatible Endpoints ist Tracing nicht zwingend verfügbar.
# Bei direkter Nutzung der OpenAI Platform kann diese Zeile entfernt werden,
# wenn OpenAI Tracing verwendet werden soll.
set_tracing_disabled(True)

print(f"MCP URL: {MCP_URL}")
print(f"Model:   {AI_MODEL}")

---

### Read-only MCP-Toolset

Das Agents SDK kann MCP-Tools direkt filtern. Damit muss keine eigene MCP→OpenAI-Tool-Konvertierung mehr implementiert werden.


In [ ]:
READ_ONLY_TOOLS = [
    "catalog_list_items",
    "catalog_get_item",
    "customer_list_items",
    "customer_get_item",
    "order_list_items",
    "order_get_item",
]

read_only_filter = create_static_tool_filter(
    allowed_tool_names=READ_ONLY_TOOLS,
)


### MCP-Server für den Agenten

`MCPServerStreamableHttp` verwaltet die MCP-Verbindung. `cache_tools_list=True` verhindert unnötige `list_tools()`-Roundtrips, solange sich das Toolset während eines Runs nicht verändert.


In [ ]:
autoshop_mcp = MCPServerStreamableHttp(
    name="AutoShop MCP",
    params={
        "url": MCP_URL,
        "timeout": 30,
    },
    cache_tools_list=True,
    tool_filter=read_only_filter,
    max_retry_attempts=2,
)


---

## Agent Skills: Dateien anlegen

Ein Agent Skill besteht mindestens aus einem Verzeichnis mit `SKILL.md` (YAML-Frontmatter und Markdown-Anleitung). `references/`, `assets/` und `scripts/` sind optional.

Die folgende Zelle erstellt drei exemplarische Skills im Benutzerverzeichnis `~/data/autoshop-agent-skills/`. **Bereits vorhandene Skill-Dateien werden nicht überschrieben**, damit Anpassungen bei erneutem Ausführen erhalten bleiben. Die Inhalte lassen sich danach unabhängig vom Python-Code ändern.

In [ ]:
from pathlib import Path

SKILLS_ROOT = Path.home() / "data" / "autoshop-agent-skills"

SKILL_FILES = {
    "vehicle-recommendation/SKILL.md": """---
name: vehicle-recommendation
description: Empfiehlt passende Fahrzeuge auf Basis von AutoShop-Catalog-Daten. Verwenden, wenn Kunden ein Auto suchen oder Fahrzeuge vergleichen möchten.
---
# Fahrzeugempfehlung

1. Analysiere Budget, Platzbedarf und Anforderungen des Kunden.
2. Suche passende Fahrzeuge ausschliesslich über die freigegebenen AutoShop-MCP-Tools.
3. Vergleiche die gelieferten Eigenschaften und Preise.
4. Wähle ein passendes Fahrzeug aus und begründe den Entscheid mit echten Catalog-Daten.
5. Gib die wichtigsten Fakten, die Catalog-ID und den Preis strukturiert zurück.

Fehlende Daten als fehlend kennzeichnen. Keine Fahrzeuge, IDs, Preise oder Verfügbarkeiten erfinden.
""",
    "vehicle-enrichment/SKILL.md": """---
name: vehicle-enrichment
description: Ergänzt allgemeine Fahrzeugvorteile, mögliche Extras und Entscheidungshilfen. Verwenden, wenn bereits interne AutoShop-Fahrzeugdaten vorliegen.
---
# Fahrzeug-Enrichment

1. Verwende die übergebenen internen AutoShop-Fahrzeugdaten nur als Kontext.
2. Erläutere allgemeine Vorteile der jeweiligen Fahrzeugklasse.
3. Schlage passende Extras vor und kennzeichne diese ausdrücklich als unverbindlich.
4. Nenne Aspekte für die Kaufentscheidung, ohne konkrete Ausstattungen zu behaupten.

Keine internen IDs, Shoppreise, konkreten Fahrzeuge oder Verfügbarkeiten hinzuerfinden.
""",
    "offer-generation/SKILL.md": """---
name: offer-generation
description: Erstellt eine deutschsprachige Markdown-Offerte aus echten AutoShop-Fahrzeugdaten und unverbindlichen Zusatzinformationen. Verwenden, wenn eine Kundenofferte verlangt wird.
---
# Markdown-Offerte erstellen

1. Lies zuerst `references/offer-template.md` mit dem Tool `read_skill_reference`.
2. Verwende das dort definierte Markdown-Gerüst fuer die Offerte.
3. Übernimm Fahrzeuge, IDs und Preise ausschliesslich aus dem internen AutoShop-Resultat.
4. Kennzeichne Extras aus dem externen Agenten als unverbindliche Vorschlaege.
5. Formuliere auf Deutsch und weise auf den unverbindlichen Education-Charakter hin.

Keine zusätzlichen Preise oder Shop-Verfügbarkeiten erfinden.
""",
    "offer-generation/references/offer-template.md": """# Offerte

## Kunde

## Kundenwunsch

## Empfohlenes Fahrzeug

## Preis gemäss AutoShop-Catalog

## Empfohlene Extras

## Begründung

## Hinweis

Diese Offerte ist ein Education-Beispiel und nicht rechtsverbindlich.
""",
}

for relative_path, content in SKILL_FILES.items():
    path = SKILLS_ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        path.write_text(content, encoding="utf-8")

print("Skill-Dateien:")
for path in sorted(SKILLS_ROOT.rglob("*.md")):
    print("-", path.relative_to(SKILLS_ROOT))

### Skill Discovery und Activation

**Discovery:** Zunächst werden ausschliesslich `name` und `description` aus der YAML-Frontmatter gelesen und in die Agent-Instructions übernommen.

**Activation:** Über `load_skill(name)` kann der Agent die vollständige `SKILL.md` laden. Weiterführende Dateien werden bei Bedarf durch `read_skill_reference(name, relative_path)` gelesen. Beide Funktionen sind SDK-`function_tool`s; sie führen keine Skill-Skripte aus und erlauben keine Zugriffe ausserhalb der jeweiligen erlaubten Skill-Verzeichnisse.

Das zeigt die drei Stufen des Agent-Skills-Standards ohne Wechsel auf `SandboxAgent`.

In [ ]:
import yaml
from agents import function_tool


def skill_directory(name: str) -> Path:
    """Validiert einen Skill-Namen und liefert dessen lokales Verzeichnis."""
    if not name or any(ch not in "abcdefghijklmnopqrstuvwxyz0123456789-" for ch in name):
        raise ValueError("Ungültiger Skill-Name")
    directory = (SKILLS_ROOT / name).resolve()
    if not directory.is_relative_to(SKILLS_ROOT.resolve()):
        raise ValueError("Pfad ausserhalb des Skill-Verzeichnisses")
    return directory


def split_skill_file(name: str) -> tuple[dict, str]:
    """Parst YAML-Frontmatter und Anleitung, ohne andere Skill-Dateien zu laden."""
    skill_file = (skill_directory(name) / "SKILL.md").resolve()
    if not skill_file.is_relative_to(skill_directory(name)):
        raise ValueError("SKILL.md verweist auf einen nicht erlaubten Pfad")
    raw = skill_file.read_text(encoding="utf-8")
    if not raw.startswith("---\n") or "\n---\n" not in raw[4:]:
        raise ValueError(f"SKILL.md von {name} hat keine gültige Frontmatter")
    frontmatter, instructions = raw[4:].split("\n---\n", 1)
    meta = yaml.safe_load(frontmatter)
    if not isinstance(meta, dict) or meta.get("name") != name or not meta.get("description"):
        raise ValueError(f"Ungültige Skill-Metadaten für {name}")
    return meta, instructions.strip()


def skill_index(names: tuple[str, ...]) -> str:
    """Discovery: nur Metadaten; der SKILL.md-Inhalt bleibt ungeladen."""
    return "\n".join(
        f"- {name}: {split_skill_file(name)[0]['description']}"
        for name in names
    )


def skill_tools(allowed_names: tuple[str, ...]):
    """Jeder Agent sieht nur die für seine Aufgabe freigegebenen Skills."""
    allowed = set(allowed_names)

    @function_tool
    def load_skill(name: str) -> str:
        """Lädt die vollständigen Anweisungen eines verfügbaren Skills bei Bedarf."""
        if name not in allowed:
            return "Dieser Skill ist für diesen Agenten nicht freigegeben."
        _, instructions = split_skill_file(name)
        return instructions

    @function_tool
    def read_skill_reference(name: str, relative_path: str) -> str:
        """Liest eine ergänzende Markdown-Datei im references/-Ordner des Skills."""
        if name not in allowed:
            return "Dieser Skill ist für diesen Agenten nicht freigegeben."
        root = skill_directory(name)
        references = (root / "references").resolve()
        target = (root / relative_path).resolve()
        if (not target.is_relative_to(references)
                or target.suffix.lower() != ".md"
                or not target.is_file()):
            return "Nur vorhandene Markdown-Dateien unter references/ sind lesbar."
        return target.read_text(encoding="utf-8")

    return [load_skill, read_skill_reference]


INTERNAL_SKILLS = ("vehicle-recommendation",)
EXTERNAL_SKILLS = ("vehicle-enrichment",)
OFFER_SKILLS = ("offer-generation",)

for label, names in [
    ("Interner Agent", INTERNAL_SKILLS),
    ("Externer Agent", EXTERNAL_SKILLS),
    ("Offert-Agent", OFFER_SKILLS),
]:
    print(f"{label}:\n{skill_index(names)}\n")

---

## Agenten mit Skills

### 1. Interner AutoShop-Agent

Nur dieser Agent besitzt MCP-Zugriff. Er entdeckt den Skill `vehicle-recommendation` anhand seiner Metadaten und lädt die vollständigen Anweisungen über `load_skill` erst für die Empfehlung.

In [ ]:
internal_car_agent = Agent(
    name="AutoShop Internal Car Agent",
    model=AI_MODEL,
    instructions=f"""
Du bist der interne AutoShop-Verkaufsagent und darfst nur echte Catalog-Daten verwenden.

Verfügbare Skills (nur Name und Beschreibung):
{skill_index(INTERNAL_SKILLS)}

Lade vor einer Fahrzeugempfehlung den passenden Skill mit `load_skill` und
folge seinen Anweisungen. Nutze die freigegebenen MCP-Tools für alle Shopdaten.
Erfinde keine Fahrzeuge, IDs, Preise oder Verfügbarkeiten. Antworte auf Deutsch.
""",
    tools=skill_tools(INTERNAL_SKILLS),
    mcp_servers=[autoshop_mcp],
)

### 2. Externer Enrichment-Agent

Dieser Agent hat **keinen MCP-Zugriff**. Sein Skill `vehicle-enrichment` beschreibt, wie allgemeine Zusatzinformationen ohne erfundene Shop-Fakten ergänzt werden.

In [ ]:
external_enrichment_agent = Agent(
    name="External Car Enrichment Agent",
    model=AI_MODEL,
    instructions=f"""
Du ergänzt ausschliesslich allgemeine Informationen, ohne MCP-Zugriff.

Verfügbare Skills (nur Name und Beschreibung):
{skill_index(EXTERNAL_SKILLS)}

Lade vor dem Enrichment den passenden Skill mit `load_skill`.
Erfinde keine Shop-Fahrzeuge, Preise, IDs oder Verfügbarkeiten.
Extras sind unverbindliche Vorschläge. Antworte auf Deutsch.
""",
    tools=skill_tools(EXTERNAL_SKILLS),
)

### 3. Offert-Agent

Der Offert-Agent erhält nur die ermittelten Resultate. Der Skill `offer-generation` definiert den Ablauf und verweist auf `references/offer-template.md`. Die konkrete Markdown-Struktur steht damit nicht mehr im Python-Prompt.

In [ ]:
offer_agent = Agent(
    name="AutoShop Offer Agent",
    model=AI_MODEL,
    instructions=f"""
Du erstellst eine Demo-Offerte als Markdown auf Deutsch.

Verfügbare Skills (nur Name und Beschreibung):
{skill_index(OFFER_SKILLS)}

Lade vor dem Schreiben den passenden Skill mit `load_skill` und lies bei Bedarf
mit `read_skill_reference` die referenzierte Markdown-Vorlage.
Fahrzeuge, IDs und Preise stammen ausschliesslich aus den übergebenen internen
AutoShop-Daten. Externe Extras sind unverbindlich.
Die Offerte ist ein Education-Beispiel und nicht rechtsverbindlich.
""",
    tools=skill_tools(OFFER_SKILLS),
)

---

## Vollständiger Multi-Agent-Ablauf

Der MCP-Server wird für den gesamten Workflow einmal verbunden. Der `Runner` übernimmt bei jedem Agenten den Modell-/Tool-Loop.


In [ ]:
async def car_sales_agent(customer_name: str, user_request: str):
    async with autoshop_mcp:
        print("1. Interner Agent sucht AutoShop-Daten über MCP ...")
        internal_run = await Runner.run(
            internal_car_agent,
            f"""
Kundenwunsch:
{user_request}

Suche passende Fahrzeuge im AutoShop und gib eine begründete Empfehlung ab.
""",
        )
        internal_result = internal_run.final_output

        print("2. Externer Agent ergänzt allgemeine Zusatzinformationen ...")
        external_run = await Runner.run(
            external_enrichment_agent,
            f"""
Kundenwunsch:
{user_request}

Interne AutoShop-Daten:
{internal_result}

Ergänze:
- mögliche sinnvolle Extras,
- typische Vorteile dieses Fahrzeugtyps,
- Punkte, auf die der Kunde achten sollte,
- kurze Verkaufsargumente.
""",
        )
        external_result = external_run.final_output

        print("3. Offert-Agent erstellt die Markdown-Offerte ...")
        offer_run = await Runner.run(
            offer_agent,
            f"""
Erstelle eine Markdown-Offerte.

Kunde:
{customer_name}

Kundenwunsch:
{user_request}

Interne AutoShop-Daten:
{internal_result}

Externe Zusatzinformationen:
{external_result}

Lade den Offert-Skill und verwende dessen referenzierte Markdown-Vorlage
für Struktur und Hinweistext.
""",
        )

    return {
        "internal_result": internal_result,
        "external_result": external_result,
        "offer": offer_run.final_output,
        "tool_calls": {
            "intern": [getattr(item.raw_item, "name", "") for item in internal_run.new_items if item.type == "tool_call_item"],
            "extern": [getattr(item.raw_item, "name", "") for item in external_run.new_items if item.type == "tool_call_item"],
            "offerte": [getattr(item.raw_item, "name", "") for item in offer_run.new_items if item.type == "tool_call_item"],
        },
    }


---

## Aufruf


In [ ]:
from IPython.display import Markdown, display

result = await car_sales_agent(
    customer_name="Max Muster",
    user_request="Ich suche ein günstiges Auto mit viel Platz für die Familie.",
)

display(Markdown(result["offer"]))


### Optional: Zwischenresultate anzeigen


In [ ]:
display(Markdown("## Internes Agentenresultat"))
display(Markdown(result["internal_result"]))

display(Markdown("## Externes Enrichment"))
display(Markdown(result["external_result"]))


### Optional: Geladene Skills im Agentenlauf prüfen

Das Agents SDK erfasst Tool-Aufrufe in `new_items`. Die folgende Ausgabe zeigt, welche Tools während der drei Runs aufgerufen wurden. So lässt sich Discovery gegenüber Activation im Unterricht nachvollziehen.

In [ ]:
from IPython.display import Markdown, display

rows = [
    (agent, tool_name)
    for agent, tool_names in result["tool_calls"].items()
    for tool_name in tool_names
]
if rows:
    table = "| Agent | Tool-Aufruf |\n|---|---|\n"
    table += "\n".join(f"| {agent} | `{name}` |" for agent, name in rows)
    display(Markdown(table))
else:
    print("Keine Tool-Aufrufe protokolliert")

## Was sich gegenüber der ursprünglichen Variante geändert hat

| Bisheriges Agents-SDK-Notebook | Zusätzlich mit Agent Skills |
|---|---|
| Fachliche Anweisungen in `Agent.instructions` | Fachlicher Workflow in `SKILL.md` |
| Feste Markdown-Gliederung im Python-Prompt | Referenzdatei `offer-template.md` |
| Vollständige Instructions immer im Kontext | Metadaten zuerst; Anleitung via `load_skill` bei Bedarf |
| `Agent` mit MCP-Read-only-Tools | derselbe `Agent` mit MCP und `function_tool`s |
| `Runner.run()` führt den Workflow aus | unverändert |

**Ablauf:** Kundenwunsch → interner Agent + Skill + MCP → externer Agent + Skill → Offert-Agent + Skill/Referenz → Markdown-Offerte.

Das ist eine **eigene, leichte Implementation des Agent-Skills-Dateiformats**, nicht die native `Skills`-Capability eines `SandboxAgent`.

## Hinweise

- `order_delete` bleibt durch die MCP-Allowlist für den internen Agenten gesperrt.
- Externer und Offert-Agent erhalten weiterhin keinen MCP-Server.
- Die drei Skills werden lokal in `~/data/autoshop-agent-skills/` erstellt. Änderungen an vorhandenen Dateien werden durch die Erstellungszelle nicht überschrieben.
- Die Tool-Funktionen lesen nur freigegebene Skill-Dateien. `scripts/` werden in diesem Demonstrator nicht ausgeführt.
- Der Modell-Endpoint muss Tool-/Function-Calling unterstützen (wie schon für MCP). Das Notebook geht weiterhin von einer Responses-kompatiblen API aus.
- **Alternative:** Für native SDK-Skills kann ein `SandboxAgent` mit `Capabilities.default() + [Skills(...)]` eingesetzt werden. Das erfordert eine zusätzliche Sandbox-Konfiguration und ist für diesen lesenden AutoShop-Workflow nicht nötig.

**Quellen:** [Agent Skills Specification](https://agentskills.io/specification) · [OpenAI Agents SDK: Sandbox Agents](https://developers.openai.com/api/docs/guides/agents/sandboxes)